# Trabajo práctico: búsqueda voraz y A\* sobre un grafo dirigido

**Estudiante:** Nicolás Cerdá
**Curso:** Inteligencia Artificial — Semana 3, Clase 3
**Modalidad:** individual
**Entregable:** notebook ejecutable de punta a punta

---

## Propósito

Implementar tres estrategias de búsqueda sobre el grafo dirigido de la clase —costo uniforme (UCS),
voraz por el mejor primero y A\*— y comparar **qué camino devuelven, cuánto cuesta y cuántos estados expanden**.

El objetivo no es "hacer andar" un algoritmo: es **distinguir rapidez de garantía**. Sobre este grafo
voraz y A\* llegan al mismo resultado; la diferencia está en qué propiedad cada uno *puede prometer*.

**Dependencias:** ninguna externa. Solo biblioteca estándar (`heapq`, `itertools`, `math`), para que el
notebook sea reproducible en cualquier máquina.

## 1. El grafo dirigido y la heurística

El problema formal es $P=(S,A,T,s_0,G,c)$ con estados $\{S,A,B,C,D,G\}$, inicio $S$, objetivo $\{G\}$,
transiciones las aristas listadas y costo $c$ el valor de cada arista.

Las aristas son **dirigidas**: `S→A` permite ir de S a A, pero no de A a S. Por eso el grafo se
representa como un diccionario de listas de adyacencia, y nunca se agrega la arista inversa.

**Regla de desempate:** ante prioridades iguales se extrae el que se insertó antes (FIFO por orden de
inserción). La respetan los tres algoritmos.

In [1]:
import heapq
from itertools import count
from math import inf

# Aristas dirigidas: estado -> [(sucesor, costo), ...]
ARISTAS = {
    "S": [("A", 2), ("B", 2)],
    "A": [("C", 2), ("D", 5)],
    "B": [("D", 2)],
    "C": [("G", 3)],
    "D": [("G", 6)],
    "G": [],
}

# Heurística: estimación del costo restante hasta G
H = {"S": 7, "A": 5, "B": 7, "C": 3, "D": 6, "G": 0}

INICIAL = "S"
OBJETIVOS = {"G"}

# Orden declarado (el de la consigna), no alfabético: mantiene las tablas legibles.
ESTADOS = ["S", "A", "B", "C", "D", "G"]
assert set(ESTADOS) == set(ARISTAS) == set(H), "estados, aristas y heurística deben coincidir"

print("Aristas dirigidas y costos")
print("-" * 34)
for origen in ESTADOS:
    for destino, costo in ARISTAS[origen]:
        print(f"  {origen} -> {destino}   costo {costo}")

print()
print("Heurística h(n)")
print("-" * 34)
print("  estado " + "  ".join(f"{e:>3}" for e in ESTADOS))
print("  h(n)   " + "  ".join(f"{H[e]:>3}" for e in ESTADOS))

Aristas dirigidas y costos
----------------------------------
  S -> A   costo 2
  S -> B   costo 2
  A -> C   costo 2
  A -> D   costo 5
  B -> D   costo 2
  C -> G   costo 3
  D -> G   costo 6

Heurística h(n)
----------------------------------
  estado   S    A    B    C    D    G
  h(n)     7    5    7    3    6    0


### 1.1 Auditoría de la heurística

Antes de usar $h$ hay que declarar qué garantiza. Dos propiedades distintas:

- **Admisible:** $0 \le h(n) \le h^*(n)$ — nunca sobreestima el costo óptimo restante.
- **Consistente:** $h(n) \le c(n,n') + h(n')$ para toda arista — la estimación no "salta" entre vecinos.

La consistencia implica admisibilidad (con $h(G)=0$ y $h \ge 0$), y es la condición que determina si A\*
necesita reaperturas. La auditoría se calcula, no se afirma: $h^*$ se obtiene con Dijkstra sobre el
**grafo invertido** desde G.

In [2]:
def costos_optimos_hasta(objetivos, aristas):
    # h*(n): costo óptimo real de n hasta un objetivo. Dijkstra sobre el grafo invertido.
    inverso = {e: [] for e in aristas}
    for origen, sucesores in aristas.items():
        for destino, costo in sucesores:
            inverso[destino].append((origen, costo))

    dist = {e: inf for e in aristas}
    cola = []
    for objetivo in objetivos:
        dist[objetivo] = 0
        heapq.heappush(cola, (0, objetivo))

    while cola:
        d, nodo = heapq.heappop(cola)
        if d > dist[nodo]:
            continue
        for previo, costo in inverso[nodo]:
            if d + costo < dist[previo]:
                dist[previo] = d + costo
                heapq.heappush(cola, (dist[previo], previo))
    return dist


H_ESTRELLA = costos_optimos_hasta(OBJETIVOS, ARISTAS)

print("Admisibilidad:  0 <= h(n) <= h*(n)")
print("-" * 48)
print(f"  {'estado':<8}{'h(n)':>6}{'h*(n)':>8}   veredicto")
admisible = True
for e in ESTADOS:
    ok = 0 <= H[e] <= H_ESTRELLA[e]
    admisible &= ok
    marca = "ok" if ok else "VIOLACION"
    exacta = "  (exacta)" if H[e] == H_ESTRELLA[e] else ""
    print(f"  {e:<8}{H[e]:>6}{H_ESTRELLA[e]:>8}   {marca}{exacta}")
print()
print(f"  => h es {'ADMISIBLE' if admisible else 'NO admisible'} en este grafo")

print()
print("Consistencia:  h(n) <= c(n,n') + h(n')")
print("-" * 48)
consistente = True
for origen in ESTADOS:
    for destino, costo in ARISTAS[origen]:
        ok = H[origen] <= costo + H[destino]
        consistente &= ok
        marca = "ok" if ok else "VIOLACION"
        igualdad = "  (con igualdad)" if H[origen] == costo + H[destino] else ""
        suma = costo + H[destino]
        print(f"  {origen} -> {destino}:  {H[origen]} <= {costo} + {H[destino]} = {suma}   {marca}{igualdad}")
print()
print(f"  => h es {'CONSISTENTE' if consistente else 'NO consistente'} en este grafo")
print()
print("  Cero violaciones demuestra consistencia en este grafo revisado, no en datos futuros.")

Admisibilidad:  0 <= h(n) <= h*(n)
------------------------------------------------
  estado    h(n)   h*(n)   veredicto
  S            7       7   ok  (exacta)
  A            5       5   ok  (exacta)
  B            7       8   ok
  C            3       3   ok  (exacta)
  D            6       6   ok  (exacta)
  G            0       0   ok  (exacta)

  => h es ADMISIBLE en este grafo

Consistencia:  h(n) <= c(n,n') + h(n')
------------------------------------------------
  S -> A:  7 <= 2 + 5 = 7   ok  (con igualdad)
  S -> B:  7 <= 2 + 7 = 9   ok
  A -> C:  5 <= 2 + 3 = 5   ok  (con igualdad)
  A -> D:  5 <= 5 + 6 = 11   ok
  B -> D:  7 <= 2 + 6 = 8   ok
  C -> G:  3 <= 3 + 0 = 3   ok  (con igualdad)
  D -> G:  6 <= 6 + 0 = 6   ok  (con igualdad)

  => h es CONSISTENTE en este grafo

  Cero violaciones demuestra consistencia en este grafo revisado, no en datos futuros.


## 2. Estructura de datos y esqueleto común

Cada nodo conserva `(estado, padre, acción, g, h, f)`, donde `g` es el costo recorrido, `h` la
estimación restante y `f` la prioridad. El campo `padre` es la referencia que permite **reconstruir el
camino desde los padres** al final, en lugar de escribirlo a mano.

Los tres algoritmos comparten el mismo esqueleto y **solo cambia la clave de prioridad**:

| Algoritmo | Prioridad | Qué privilegia |
|---|---|---|
| UCS | $g$ | el camino más barato recorrido |
| Voraz | $h$ | el estado que parece más cerca del objetivo |
| A\* | $g+h$ | el costo total estimado de la solución |

Decisiones de implementación que la consigna exige:

1. **Cola de prioridad con `heapq`** y desempate estable por orden de inserción: se empuja la tupla
   `(prioridad, orden, nodo)`, donde `orden` viene de un contador monótono. Como `orden` nunca se
   repite, ante prioridades iguales gana el insertado antes y los diccionarios nunca se comparan entre sí.
2. **El objetivo se comprueba al extraer, no al generar.** Comprobarlo al generar devolvería el primer
   camino encontrado, que no tiene por qué ser el más barato.
3. **Descarte de entradas obsoletas:** si el `g` del nodo extraído es peor que `mejor_g[estado]`, esa
   entrada quedó vieja por una mejora posterior y se saltea.
4. **Reapertura** = cada vez que mejora `mejor_g` de un estado ya conocido.

> **Nota sobre el descarte de obsoletos en voraz.** El pseudocódigo de VORAZ de la consigna no
> explicita la línea de descarte, pero el enunciado dice que los tres algoritmos comparten el control
> de repetidos. Acá se aplica de forma uniforme. En este grafo la decisión no cambia ningún resultado:
> voraz nunca llega a extraer una entrada obsoleta, como se ve en su traza.

In [3]:
def crear_nodo(estado, padre=None, accion=None, g=0, h=0):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "g": g,
        "h": h,
        "f": g + h,
    }


def reconstruir_camino(nodo):
    # El camino NO se escribe a mano: se rearma siguiendo los padres hacia atrás.
    camino = []
    actual = nodo
    while actual is not None:
        camino.append(actual["estado"])
        actual = actual["padre"]
    camino.reverse()
    return camino


# Lo único que distingue a los tres algoritmos.
PRIORIDAD = {
    "UCS":   lambda g, h: g,
    "Voraz": lambda g, h: h,
    "A*":    lambda g, h: g + h,
}


def formatear_frontera(frontera):
    # Contenido de la frontera con su prioridad, en el orden en que se extraería.
    if not frontera:
        return "(vacia)"
    entradas = sorted(frontera, key=lambda t: (t[0], t[1]))
    return " ".join(f"{nodo['estado']}({p})" for p, _, nodo in entradas)

In [4]:
def buscar(nombre, h=None, aristas=None, inicial=INICIAL, objetivos=OBJETIVOS):
    h = H if h is None else h
    aristas = ARISTAS if aristas is None else aristas
    prioridad = PRIORIDAD[nombre]
    orden = count()          # desempate estable: FIFO por orden de inserción
    frontera = []
    mejor_g = {inicial: 0}
    traza = []
    generados = expandidos = reaperturas = obsoletos = 0

    raiz = crear_nodo(inicial, g=0, h=h[inicial])
    heapq.heappush(frontera, (prioridad(0, h[inicial]), next(orden), raiz))
    generados += 1
    frontera_max = 1

    while frontera:
        antes = formatear_frontera(frontera)          # foto previa a la extracción
        p, _, nodo = heapq.heappop(frontera)
        estado = nodo["estado"]

        # 1) descartar entradas obsoletas
        if nodo["g"] > mejor_g.get(estado, inf):
            obsoletos += 1
            nota = f"descartado: mejor_g[{estado}]={mejor_g[estado]}"
            traza.append((len(traza) + 1, f"{estado} (g={nodo['g']}, obsoleto)", p, antes, nota))
            continue

        # 2) prueba del objetivo AL EXTRAER
        if estado in objetivos:
            traza.append((len(traza) + 1, estado, p, antes, "objetivo alcanzado"))
            return {
                "algoritmo": nombre,
                "camino": reconstruir_camino(nodo),
                "costo": nodo["g"],
                "generados": generados,
                "expandidos": expandidos,
                "frontera_max": frontera_max,
                "reaperturas": reaperturas,
                "obsoletos": obsoletos,
                "traza": traza,
            }

        # 3) expandir: relajar cada transición legal
        expandidos += 1
        detalle = []
        for sucesor, costo in aristas.get(estado, []):
            nuevo_g = nodo["g"] + costo
            if nuevo_g < mejor_g.get(sucesor, inf):
                if sucesor in mejor_g:
                    reaperturas += 1
                    detalle.append(f"{sucesor}: g {mejor_g[sucesor]}->{nuevo_g} (REAPERTURA)")
                else:
                    detalle.append(f"{sucesor}: g={nuevo_g}")
                mejor_g[sucesor] = nuevo_g
                accion = f"{estado} -> {sucesor}"
                hijo = crear_nodo(sucesor, padre=nodo, accion=accion, g=nuevo_g, h=h[sucesor])
                heapq.heappush(frontera, (prioridad(nuevo_g, h[sucesor]), next(orden), hijo))
                generados += 1
            else:
                detalle.append(f"{sucesor}: g={nuevo_g} sin mejora")

        frontera_max = max(frontera_max, len(frontera))
        resumen = "expande -> " + ("; ".join(detalle) if detalle else "sin sucesores")
        traza.append((len(traza) + 1, estado, p, antes, resumen))

    return {
        "algoritmo": nombre, "camino": None, "costo": inf, "generados": generados,
        "expandidos": expandidos, "frontera_max": frontera_max,
        "reaperturas": reaperturas, "obsoletos": obsoletos, "traza": traza,
    }

## 3. Ejecución y trazas

Por cada expansión se registra el estado extraído, su prioridad, el contenido de la frontera **antes de
extraer** (con la prioridad de cada entrada, en el orden en que se extraerían) y qué produjo la
relajación de sus sucesores.

In [5]:
CLAVE = {"UCS": "g", "Voraz": "h", "A*": "f = g + h"}


def mostrar(resultado):
    nombre = resultado["algoritmo"]
    print("=" * 104)
    print(f"{nombre}  --  prioridad por {CLAVE.get(nombre, '?')}")
    print("=" * 104)
    cab = f"{'#':<3}{'extraido':<24}{'prio':>5}   {'frontera antes de extraer':<34}resultado"
    print(cab)
    print("-" * 104)
    for paso, extraido, p, antes, nota in resultado["traza"]:
        print(f"{paso:<3}{extraido:<24}{p:>5}   {antes:<34}{nota}")
    print("-" * 104)
    camino = " -> ".join(resultado["camino"]) if resultado["camino"] else "FRACASO"
    print(f"Camino: {camino}")
    print(f"Costo: {resultado['costo']}")
    print(f"Generados: {resultado['generados']}   "
          f"Expandidos: {resultado['expandidos']}   "
          f"Frontera maxima: {resultado['frontera_max']}   "
          f"Reaperturas: {resultado['reaperturas']}   "
          f"Obsoletos descartados: {resultado['obsoletos']}")
    print()


ucs = buscar("UCS")
mostrar(ucs)

UCS  --  prioridad por g
#  extraido                 prio   frontera antes de extraer         resultado
--------------------------------------------------------------------------------------------------------
1  S                           0   S(0)                              expande -> A: g=2; B: g=2
2  A                           2   A(2) B(2)                         expande -> C: g=4; D: g=7
3  B                           2   B(2) C(4) D(7)                    expande -> D: g 7->4 (REAPERTURA)
4  C                           4   C(4) D(4) D(7)                    expande -> G: g=7
5  D                           4   D(4) D(7) G(7)                    expande -> G: g=10 sin mejora
6  D (g=7, obsoleto)           7   D(7) G(7)                         descartado: mejor_g[D]=4
7  G                           7   G(7)                              objetivo alcanzado
--------------------------------------------------------------------------------------------------------
Camino: S -> A -> C -> G


In [6]:
voraz = buscar("Voraz")
mostrar(voraz)

Voraz  --  prioridad por h
#  extraido                 prio   frontera antes de extraer         resultado
--------------------------------------------------------------------------------------------------------
1  S                           7   S(7)                              expande -> A: g=2; B: g=2
2  A                           5   A(5) B(7)                         expande -> C: g=4; D: g=7
3  C                           3   C(3) D(6) B(7)                    expande -> G: g=7
4  G                           0   G(0) D(6) B(7)                    objetivo alcanzado
--------------------------------------------------------------------------------------------------------
Camino: S -> A -> C -> G
Costo: 7
Generados: 6   Expandidos: 3   Frontera maxima: 3   Reaperturas: 0   Obsoletos descartados: 0



In [7]:
a_estrella = buscar("A*")
mostrar(a_estrella)

A*  --  prioridad por f = g + h
#  extraido                 prio   frontera antes de extraer         resultado
--------------------------------------------------------------------------------------------------------
1  S                           7   S(7)                              expande -> A: g=2; B: g=2
2  A                           7   A(7) B(9)                         expande -> C: g=4; D: g=7
3  C                           7   C(7) B(9) D(13)                   expande -> G: g=7
4  G                           7   G(7) B(9) D(13)                   objetivo alcanzado
--------------------------------------------------------------------------------------------------------
Camino: S -> A -> C -> G
Costo: 7
Generados: 6   Expandidos: 3   Frontera maxima: 3   Reaperturas: 0   Obsoletos descartados: 0



## 4. Verificación

Antes de sacar conclusiones conviene comprobar que los caminos devueltos son **legales** (usan solo
aristas existentes y en el sentido correcto) y que el costo informado coincide con la suma de las
aristas recorridas. Si el camino se hubiera "escrito a mano", esta celda lo delataría.

In [8]:
def verificar(resultado, aristas=None, inicial=INICIAL, objetivos=OBJETIVOS):
    aristas = ARISTAS if aristas is None else aristas
    camino = resultado["camino"]
    problemas = []
    if not camino:
        return ["no devolvio camino"]
    if camino[0] != inicial:
        problemas.append(f"no arranca en {inicial}")
    if camino[-1] not in objetivos:
        problemas.append("no termina en un objetivo")
    total = 0
    for origen, destino in zip(camino, camino[1:]):
        arista = [cst for dst, cst in aristas.get(origen, []) if dst == destino]
        if not arista:
            problemas.append(f"la arista {origen}->{destino} no existe en ese sentido")
        else:
            total += arista[0]
    if total != resultado["costo"]:
        problemas.append(f"costo informado {resultado['costo']} != suma real {total}")
    return problemas


RESULTADOS = [ucs, voraz, a_estrella]

# Costo óptimo real, calculado de forma independiente a los tres algoritmos.
optimo = H_ESTRELLA[INICIAL]
print(f"Costo optimo real desde {INICIAL} (Dijkstra independiente): {optimo}")
print()

for r in RESULTADOS:
    fallas = verificar(r)
    estado = "OK" if not fallas else "FALLA: " + "; ".join(fallas)
    marca = "optimo" if r["costo"] == optimo else f"SUBOPTIMO (+{r['costo'] - optimo})"
    print(f"  {r['algoritmo']:<6} camino legal y costo coherente: {estado:<10} costo {r['costo']} -> {marca}")

Costo optimo real desde S (Dijkstra independiente): 7

  UCS    camino legal y costo coherente: OK         costo 7 -> optimo
  Voraz  camino legal y costo coherente: OK         costo 7 -> optimo
  A*     camino legal y costo coherente: OK         costo 7 -> optimo


## 5. Tabla comparativa

In [9]:
NOMBRE_PRIORIDAD = {"UCS": "g", "Voraz": "h", "A*": "g + h"}

FILAS = [
    ("Camino",                        lambda r: " -> ".join(r["camino"])),
    ("Costo",                         lambda r: str(r["costo"])),
    ("Prioridad",                     lambda r: NOMBRE_PRIORIDAD[r["algoritmo"]]),
    ("Expandidos antes de extraer G", lambda r: str(r["expandidos"])),
    ("Generados",                     lambda r: str(r["generados"])),
    ("Frontera maxima",               lambda r: str(r["frontera_max"])),
    ("Reaperturas",                   lambda r: str(r["reaperturas"])),
]

ancho_etiqueta = max(len(e) for e, _ in FILAS) + 2
ancho_col = 18

encabezado = f"{'Resultado':<{ancho_etiqueta}}"
for r in RESULTADOS:
    encabezado += f"{r['algoritmo']:<{ancho_col}}"
print(encabezado)
print("-" * (ancho_etiqueta + ancho_col * len(RESULTADOS)))
for etiqueta, valor in FILAS:
    fila = f"{etiqueta:<{ancho_etiqueta}}"
    for r in RESULTADOS:
        fila += f"{valor(r):<{ancho_col}}"
    print(fila)

Resultado                      UCS               Voraz             A*                
-------------------------------------------------------------------------------------
Camino                         S -> A -> C -> G  S -> A -> C -> G  S -> A -> C -> G  
Costo                          7                 7                 7                 
Prioridad                      g                 h                 g + h             
Expandidos antes de extraer G  5                 3                 3                 
Generados                      7                 6                 6                 
Frontera maxima                3                 3                 3                 
Reaperturas                    1                 0                 0                 


## 6. Experimento de control: A\* con $h=0$

La pregunta 3 se puede *responder* o se puede *mostrar*. Como el esqueleto está parametrizado por la
heurística, alcanza con volver a correr A\* pasándole $h=0$ para todos los estados y comparar sus
métricas con las de UCS.

In [10]:
H_NULA = {e: 0 for e in ESTADOS}
a_con_h0 = buscar("A*", h=H_NULA)
a_con_h0["algoritmo"] = "A* (h=0)"

campos = ["camino", "costo", "generados", "expandidos", "frontera_max", "reaperturas"]
iguales = all(ucs[k] == a_con_h0[k] for k in campos)

print(f"{'metrica':<18}{'UCS':<26}{'A* con h=0':<26}¿igual?")
print("-" * 78)
for k in campos:
    v1 = " -> ".join(ucs[k]) if k == "camino" else str(ucs[k])
    v2 = " -> ".join(a_con_h0[k]) if k == "camino" else str(a_con_h0[k])
    print(f"{k:<18}{v1:<26}{v2:<26}{'si' if ucs[k] == a_con_h0[k] else 'NO'}")
print("-" * 78)
print()
print(f"=> A* con h=0 es {'identico a' if iguales else 'distinto de'} UCS en las seis metricas.")
print("   Con f = g + 0 = g, la prioridad de A* es exactamente la de costo uniforme.")

metrica           UCS                       A* con h=0                ¿igual?
------------------------------------------------------------------------------
camino            S -> A -> C -> G          S -> A -> C -> G          si
costo             7                         7                         si
generados         7                         7                         si
expandidos        5                         5                         si
frontera_max      3                         3                         si
reaperturas       1                         1                         si
------------------------------------------------------------------------------

=> A* con h=0 es identico a UCS en las seis metricas.
   Con f = g + 0 = g, la prioridad de A* es exactamente la de costo uniforme.


## 7. Preguntas de análisis

### 1. ¿Por qué voraz y A\* coinciden en este grafo? ¿Qué condición del grafo y de la heurística lo explica?

Coinciden porque **sobre los estados del camino óptimo la heurística es exacta**: la auditoría de la
sección 1.1 muestra $h(S)=h^*(S)=7$, $h(A)=h^*(A)=5$, $h(C)=h^*(C)=3$ y $h(G)=0$. Donde $h=h^*$, "lo que
parece más cerca" *es* lo más cerca, y por lo tanto ordenar por $h$ elige la misma rama que ordenar por
$g+h$.

El segundo ingrediente es que la alternativa engañosa está bien penalizada: B es el desvío que podría
descarrilar a voraz, pero $h(B)=7$ es el valor más alto del grafo, así que voraz nunca lo prefiere.
Además el grafo es pequeño, dirigido y acíclico, y desde A el sucesor con menor $h$ (C, con 3) es
también el que abre el camino barato.

Hay que ser preciso con lo que esto demuestra: es una **coincidencia de esta instancia**, no una
propiedad de voraz. La condición que la explica —que $h$ sea exacta sobre el camino óptimo— no se puede
suponer en general, porque calcular $h^*$ equivale a resolver el problema original. La consistencia de
$h$ (verificada arista por arista, cero violaciones) garantiza el resultado de A\*; no garantiza nada
sobre voraz.

### 2. ¿Garantiza voraz devolver el camino de menor costo en general?

**No.** La justificación está en su prioridad, no en este ejemplo: voraz ordena la frontera por
$f(n)=h(n)$ y **descarta por completo la información de $g(n)$**, el costo ya pagado. Un algoritmo que
no mira lo que gastó no puede comparar el costo total de dos alternativas, así que nada le impide
comprometerse con una rama que parece cercana al objetivo pero a la que se llegó por un camino caro.

Esto vale **aun con una heurística admisible**: la admisibilidad acota la estimación del futuro
($h \le h^*$), pero la optimalidad exige sumar pasado y futuro. El contraejemplo de la clase lo muestra:
con $S\to A$ de costo 100 y $h(A)=1$ frente a $S\to B$ de costo 1 y $h(B)=5$, voraz elige A porque
$1<5$ y devuelve un camino de costo 101, cuando existe $S\to B\to G$ de costo 6. Voraz es *completo*
sobre grafos finitos con control de repetidos, pero no *óptimo*.

### 3. ¿Qué ocurre si se usa $h=0$ en A\*? ¿Con qué algoritmo coincide entonces?

Su prioridad pasa a ser $f(n)=g(n)+0=g(n)$, es decir, exactamente la de **costo uniforme (UCS)**. A\* con
la heurística nula *es* UCS: la sección 6 lo verifica corriendo el mismo esqueleto con $h=0$ y obteniendo
las seis métricas idénticas —mismo camino, mismo costo, mismos expandidos, misma reapertura.

Vale la pena notar qué se conserva y qué se pierde. $h=0$ es trivialmente admisible y consistente
($0 \le 0 \le h^*$ y $0 \le c + 0$ para costos no negativos), así que **A\* sigue siendo óptimo**: la
garantía no depende de tener una buena heurística. Lo que se pierde es la orientación: sin información
del futuro, la búsqueda se expande en todas las direcciones baratas por igual y expande más estados,
que es justo lo que se ve al comparar sus expansiones contra las de A\* con la $h$ real.

### 4. ¿Hubo reaperturas en UCS? ¿Y en A\* y voraz? ¿Por qué?

**UCS: 1 reapertura.** Al expandir A se alcanza D con $g=7$ (por $S\to A\to D$). Después, al expandir B
—que empataba con A en $g=2$ y salió segundo por la regla de desempate— se alcanza D con $g=4$ (por
$S\to B\to D$), que mejora `mejor_g[D]`. Se registra el nuevo padre y se inserta una entrada nueva; la
vieja queda obsoleta y se descarta cuando le toca salir (visible en la traza).

**A\*: 0 reaperturas, y esto no es casualidad.** La heurística es **consistente** (auditada arista por
arista, cero violaciones), y bajo consistencia $f$ es no decreciente a lo largo de cualquier camino, de
modo que A\* extrae los estados en orden no decreciente de $f$ y **la primera vez que extrae un estado
su $g$ ya es óptimo**. Por eso no puede aparecer después un camino mejor hacia un estado ya cerrado.
Concretamente, A\* nunca llega a expandir B: lo deja en la frontera con $f=9$, mayor que el $f=7$ del
camino bueno, así que la mejora de D vía B ni siquiera se genera.

**Voraz: 0 reaperturas, pero por un motivo distinto y mucho más débil.** No es una propiedad, es un
accidente de la terminación temprana: voraz alcanza G después de expandir solo S, A y C, y nunca llega a
expandir B, que era el único que podía mejorar `mejor_g[D]`. No hubo reaperturas porque no hubo
oportunidad de que las hubiera. Nada en la prioridad $h$ impide una reapertura en otro grafo.

La distinción es la que ordena todo el TP: A\* tiene cero reaperturas **por garantía**, voraz tiene cero
**por suerte**.

### 5. ¿"Expandir menos estados" significa "camino más barato"?

**No, son dos ejes independientes.** "Expandidos" mide *trabajo*; "costo" mide *calidad de la solución*.
Este grafo lo ilustra bien justamente porque los resultados coinciden: voraz expandió menos estados que
UCS y sin embargo **ambos devolvieron el mismo camino de costo 7**. Las expansiones extra de UCS no
fueron desperdicio ni ineficiencia: son exactamente el trabajo de descartar la alternativa por B, y es
*ese* trabajo el que le permite afirmar que 7 es el mínimo. Voraz llegó al mismo número sin haber
verificado nada, y por eso no puede prometerlo.

Que la economía de voraz no es gratis se ve en el contraejemplo de la pregunta 2: ahí también expande
menos, y devuelve 101 en lugar de 6. El mismo ahorro produce, según el grafo, el resultado correcto o
uno pésimo, lo que confirma que expandir menos no es evidencia de nada sobre el costo.

La lectura correcta: **la heurística compra expansiones, no optimalidad.** Comparando A\* contra UCS
—los dos óptimos— A\* obtiene el mismo camino de costo 7 con menos expansiones: ahí sí hay una mejora
real, porque se redujo el trabajo *conservando* la garantía. Esa es la ganancia legítima de la búsqueda
informada, y es de otra naturaleza que el ahorro de voraz.

## 8. Conclusión

Sobre este grafo los tres algoritmos devuelven $S \to A \to C \to G$ con costo 7, pero por razones
distintas y con garantías distintas:

- **UCS** lo devuelve porque comparó costos: expandió de más y tuvo una reapertura, y ese trabajo es lo
  que respalda la afirmación "7 es el mínimo".
- **A\*** lo devuelve con menos expansiones y sin reaperturas, conservando la garantía de UCS gracias a
  que $h$ es consistente. Es la única mejora legítima de las tres.
- **Voraz** lo devuelve con pocas expansiones porque en este grafo $h$ resultó exacta sobre el camino
  óptimo. Cambiando los costos sin tocar el esquema, el mismo algoritmo devuelve un camino peor.

Coincidir en el resultado no es coincidir en la propiedad: **rapidez y garantía se miden por separado.**